
> This is a notebook that downloads and filters problems having both COBOL and Java submissions from Project CodeNet. It also has scripts that run safety checks to ensure that all the COBOL code is valid and compiles on GnuCOBOL. 



In [ ]:
import os
url = "https://codait-cos-dax.s3.us.cloud-object-storage.appdomain.cloud/dax-project-codenet/1.0.0/Project_CodeNet.tar.gz"
tar_path = "/content/Project_CodeNet.tar.gz"


!wget -c {url} -O {tar_path}

print(" Download completed")





> Finding Problem IDS - PIDS - that have both COBOL and Java folders




In [ ]:
import os
import tarfile
import csv
import subprocess

# =========================
# 1. PATHS
# =========================
tar_url = "https://codait-cos-dax.s3.us.cloud-object-storage.appdomain.cloud/dax-project-codenet/1.0.0/Project_CodeNet.tar.gz"
tar_path = "/content/Project_CodeNet.tar.gz"

extract_root = "DIR"
metadata_dir = "Project_CodeNet/metadata"
pid_output_file = "java_and_cobol.txt"

# =========================
# 2. DOWNLOAD TARBALL (if needed)
# =========================
if not os.path.exists(tar_path):
    print("Downloading Project CodeNet tarball (~8GB)...")
    subprocess.run([
        "wget", "-O", tar_path, tar_url
    ], check=True)
    print("Download complete:", tar_path)
else:
    print("Tarball already exists:", tar_path)

# =========================
# 3. EXTRACT ONLY METADATA (if needed)
# =========================
if not os.path.exists(metadata_dir) or len(os.listdir(metadata_dir)) == 0:
    print("\nExtracting ONLY metadata CSVs...")

    os.makedirs("/content/Project_CodeNet", exist_ok=True)

    with tarfile.open(tar_path, "r:gz") as tar:
        count = 0
        for member in tar:
            # Only metadata CSVs
            if (
                member.isfile()
                and member.name.startswith("Project_CodeNet/metadata/")
                and member.name.endswith(".csv")
            ):
                tar.extract(member, path=extract_root)
                count += 1

                if count % 100 == 0:
                    print(f"  → Extracted {count} metadata CSVs...")

    print(f" Metadata extraction complete. Total CSVs extracted: {count}")
else:
    print(f"\n Metadata already exists: {metadata_dir}")
    print(f" Existing CSV files: {len(os.listdir(metadata_dir))}")

# =========================
# 4. SCAN METADATA FOR JAVA + COBOL
# =========================
print("\n Scanning metadata for problems with BOTH JAVA and COBOL submissions...")

csv_files = sorted(
    [f for f in os.listdir(metadata_dir) if f.endswith(".csv")]
)

problem_ids = []

for idx, csv_file in enumerate(csv_files, 1):
    csv_path = os.path.join(metadata_dir, csv_file)

    has_java = False
    has_cobol = False

    try:
        with open(csv_path, "r", encoding="utf-8", errors="ignore") as f:
            reader = csv.DictReader(f)

            # Handle weird CSVs missing headers
            if reader.fieldnames is None or "language" not in reader.fieldnames:
                # Skip malformed / special metadata files
                continue

            for row in reader:
                lang = row["language"].strip().upper()

                if lang == "JAVA":
                    has_java = True
                elif lang == "COBOL":
                    has_cobol = True

                if has_java and has_cobol:
                    pid = csv_file.replace(".csv", "")
                    problem_ids.append(pid)
                    break

    except Exception as e:
        print(f" Skipping {csv_file} due to error: {e}")

    if idx % 50 == 0:
        print(f"  → Scanned {idx}/{len(csv_files)} metadata files... Found {len(problem_ids)} matches so far")

# =========================
# 5. SAVE PIDs TO TEXT FILE
# =========================
with open(pid_output_file, "w") as f:
    for pid in problem_ids:
        f.write(pid + "\n")

# =========================
# 6. FINAL SUMMARY
# =========================
print("\n DONE!")
print(f" Problems with BOTH JAVA + COBOL: {len(problem_ids)}")
print(f" Saved PID list to: {pid_output_file}")
print(" First 10 PIDs:", problem_ids[:10])




> Extracting problems that have both COBOL and Java submissions from java_and_cobol.txt (attached in the file dir here) - 340 in total








In [ ]:
import os
import tarfile
import shutil
from tqdm import tqdm


# Configuration
tar_path = 'Project_CodeNet.tar.gz'
problem_list_file = 'java_and_cobol.txt'
final_dir = 'CodeNet_Dataset'
temp_extract_dir = 'temp_extract'

# Create directories
os.makedirs(final_dir, exist_ok=True)
os.makedirs(temp_extract_dir, exist_ok=True)

# Read problem IDs
print(" Reading problem IDs...")
with open(problem_list_file, 'r') as f:
    problem_ids = set(line.strip() for line in f if line.strip())
print(f"   Found {len(problem_ids)} problems to extract")

# Folders we want
wanted_folders = {"Java", "COBOL", "input_output"}

print("\n Extracting data from tar (SINGLE PASS - optimized)...")
print("   This will take a while but is much faster than the nested loop approach\n")

# Statistics
stats = {
    'problems_found': set(),
    'java_files': 0,
    'cobol_files': 0,
    'test_files': 0,
}

with tarfile.open(tar_path, 'r:gz') as tar:
    # Single pass through ALL members
    for member in tqdm(tar.getmembers(), desc="Scanning tar", unit="files"):
        # Check if this file belongs to one of our wanted problems
        # Path format: Project_CodeNet/data/{problem_id}/{folder}/...
        parts = member.name.split('/')

        if len(parts) < 4:
            continue

        if parts[0] != "Project_CodeNet" or parts[1] != "data":
            continue

        problem_id = parts[2]

        # Only process if this is one of our wanted problems
        if problem_id not in problem_ids:
            continue

        # Check if it's in a wanted folder
        if len(parts) >= 4:
            folder_name = parts[3]

            # Handle both "input_output" and separate "input"/"output" folders
            if folder_name in ["Java", "COBOL", "input_output", "input", "output"]:
                stats['problems_found'].add(problem_id)

                # Count file types
                if folder_name == "Java":
                    stats['java_files'] += 1
                elif folder_name == "COBOL":
                    stats['cobol_files'] += 1
                elif folder_name in ["input_output", "input", "output"]:
                    stats['test_files'] += 1

                # Extract to temp directory
                try:
                    tar.extract(member, path=temp_extract_dir)
                except Exception as e:
                    print(f"  Error extracting {member.name}: {e}")

print("\n Extraction from tar complete!")
print(f"   Problems found: {len(stats['problems_found'])}/{len(problem_ids)}")
print(f"   Java files: {stats['java_files']}")
print(f"   COBOL files: {stats['cobol_files']}")
print(f"   Test files: {stats['test_files']}")


source_base = os.path.join(temp_extract_dir, "Project_CodeNet/data")

for problem_id in tqdm(stats['problems_found']"):
    src_problem_dir = os.path.join(source_base, problem_id)
    dst_problem_dir = os.path.join(final_drive_dir, problem_id)

    if not os.path.exists(src_problem_dir):
        continue

    # Create destination directory
    os.makedirs(dst_problem_dir, exist_ok=True)

    # Move each folder
    for folder in os.listdir(src_problem_dir):
        src_folder = os.path.join(src_problem_dir, folder)
        dst_folder = os.path.join(dst_problem_dir, folder)

        if os.path.isdir(src_folder):
            if os.path.exists(dst_folder):
                shutil.rmtree(dst_folder)
            shutil.move(src_folder, dst_folder)

# Cleanup temp directory
print("\n Cleaning up temporary files...")
if os.path.exists(temp_extract_dir):
    shutil.rmtree(temp_extract_dir)

print("\n COMPLETE!")
print(f"   Data saved to: {final_dir}")
print(f"   {len(stats['problems_found'])} problems extracted")

# Check for problems that weren't found
missing = problem_ids - stats['problems_found']
if missing:
    print(f"\n  Warning: {len(missing)} problems not found in tar:")
    print(f"   Sample: {list(missing)[:5]}")

    # Save missing IDs
    with open('/content/missing_problems.txt', 'w') as f:
        for pid in sorted(missing):
            f.write(pid + '\n')
    print("Full list saved to: /content/missing_problems.txt")

# Verify a sample problem
sample_problems = list(stats['problems_found'])[:3]
print(f"\n Sample verification - checking structure of {sample_problems[0]}:")
sample_dir = os.path.join(final_drive_dir, sample_problems[0])
if os.path.exists(sample_dir):
    for item in os.listdir(sample_dir):
        item_path = os.path.join(sample_dir, item)
        if os.path.isdir(item_path):
            file_count = len(os.listdir(item_path))
            print(f"   {item}/: {file_count} files")



> Safety check 1 to see all problems extracted have both COBOL and Java files




In [ ]:
import os

base_dir = 'CodeNet_Dataset'

output_file = 'problems_missing_java_cobol.txt'

bad_problems = []
print(len(os.listdir(base_dir)))
for problem_id in os.listdir(base_dir):
    problem_path = os.path.join(base_dir, problem_id)

    if not os.path.isdir(problem_path):
      print(f"  Warning: {problem_id} is not a directory")
      continue

    # Get folder names inside this problem
    folders = set(os.listdir(problem_path))
    print(folders)

    has_java = 'Java' in folders
    has_cobol = 'COBOL' in folders
    has_io = 'input.txt' in folders

    # Flag problem if it has input/output but missing BOTH Java and COBOL
    if has_io and (not has_java and not has_cobol):
        bad_problems.append(problem_id)
        print(f"  Warning: {problem_id} has input/output but missing Java & COBOL")

print(f" Problems missing Java & COBOL: {len(bad_problems)}")
print("Sample problem IDs:", bad_problems[:10])

# Save to a new text file
with open(output_file, 'w') as f:
    for pid in sorted(bad_problems):
        f.write(pid + '\n')

print(f" Saved the list to: {output_file}")




>  Safety check 2 to see that the content inside the COBOL folders actually contain COBOL code



In [ ]:
import os
from pathlib import Path
import re

class CobolValidator:
    def __init__(self, CodeNet_Dataset_path):
        """
        Initialize validator
        CodeNet_Dataset_path: Path to CodeNet_Dataset folder
        """
        self.codenet_path = Path(CodeNet_Dataset_path)

        # COBOL-specific patterns that should be present in valid COBOL code
        self.cobol_patterns = [
            r'IDENTIFICATION\s+DIVISION',
            r'PROGRAM-ID',
            r'DATA\s+DIVISION',
            r'PROCEDURE\s+DIVISION',
            r'WORKING-STORAGE\s+SECTION',
            r'ENVIRONMENT\s+DIVISION'
        ]

        # C/C++ patterns that indicate it's NOT COBOL
        self.cpp_patterns = [
            r'#include\s*<',
            r'int\s+main\s*\(',
            r'std::',
            r'cout\s*<<',
            r'cin\s*>>',
            r'printf\s*\(',
            r'scanf\s*\('
        ]

        # Java patterns
        self.java_patterns = [
            r'public\s+class\s+',
            r'public\s+static\s+void\s+main',
            r'import\s+java\.',
            r'System\.out\.println'
        ]

        # Python patterns
        self.python_patterns = [
            r'def\s+\w+\s*\(',
            r'import\s+\w+',
            r'from\s+\w+\s+import',
            r'if\s+__name__\s*==\s*["\']__main__["\']'
        ]

    def get_pid_folders(self):
        """Get all PID folders from CodeNet_Dataset"""
        pid_folders = []
        if self.codenet_path.exists():
            for item in self.codenet_path.iterdir():
                if item.is_dir():
                    pid_folders.append(item)
        return sorted(pid_folders)

    def find_cobol_folder(self, pid_folder):
        """Check if COBOL folder exists in PID folder"""
        cobol_folder = pid_folder / 'COBOL'
        if cobol_folder.exists() and cobol_folder.is_dir():
            return cobol_folder

        # Try lowercase
        cobol_folder = pid_folder / 'cobol'
        if cobol_folder.exists() and cobol_folder.is_dir():
            return cobol_folder

        return None

    def find_cobol_files(self, cobol_folder):
        """Find all .cob files in COBOL folder"""
        return list(cobol_folder.glob('*.cob')) + list(cobol_folder.glob('*.COB'))

    def is_valid_cobol(self, content):
        """Check if content is valid COBOL code"""
        # Check for COBOL-specific keywords
        cobol_matches = sum(1 for pattern in self.cobol_patterns
                           if re.search(pattern, content, re.IGNORECASE))

        # Check for other language patterns
        cpp_matches = sum(1 for pattern in self.cpp_patterns
                         if re.search(pattern, content))
        java_matches = sum(1 for pattern in self.java_patterns
                          if re.search(pattern, content))
        python_matches = sum(1 for pattern in self.python_patterns
                            if re.search(pattern, content))

        other_lang_matches = cpp_matches + java_matches + python_matches

        # Valid COBOL should have at least 2 COBOL patterns and no other language patterns
        return cobol_matches >= 2 and other_lang_matches == 0

    def detect_language(self, content):
        """Try to detect what language the file actually contains"""
        cpp_matches = sum(1 for pattern in self.cpp_patterns
                         if re.search(pattern, content))
        java_matches = sum(1 for pattern in self.java_patterns
                          if re.search(pattern, content))
        python_matches = sum(1 for pattern in self.python_patterns
                            if re.search(pattern, content))
        cobol_matches = sum(1 for pattern in self.cobol_patterns
                           if re.search(pattern, content, re.IGNORECASE))

        scores = {
            'C++': cpp_matches,
            'Java': java_matches,
            'Python': python_matches,
            'COBOL': cobol_matches
        }

        max_score = max(scores.values())
        if max_score == 0:
            return 'Unknown'

        # Return language with highest score
        return max(scores.items(), key=lambda x: x[1])[0]

    def validate_file(self, cobol_file):
        """Validate a single COBOL file"""
        try:
            with open(cobol_file, 'r', encoding='utf-8', errors='ignore') as f:
                content = f.read()

            is_valid = self.is_valid_cobol(content)
            detected_lang = self.detect_language(content)

            return {
                'valid': is_valid,
                'detected_language': detected_lang,
                'file_size': len(content),
                'preview': content[:200].replace('\n', ' ')[:100]
            }

        except Exception as e:
            return {
                'valid': False,
                'detected_language': 'Error',
                'error': str(e)
            }

    def validate_all(self, show_valid=False):
        """
        Validate all COBOL files in CodeNet_Dataset
        show_valid: If True, also print valid COBOL files
        """
        if not self.codenet_path.exists():
            print(f"Folder not found: {self.codenet_path}")
            return None

        pid_folders = self.get_pid_folders()

        if not pid_folders:
            print(f"No PID folders found in {self.codenet_path}")
            return None

        stats = {
            'total_pids': len(pid_folders),
            'has_cobol_folder': 0,
            'no_cobol_folder': 0,
            'valid_cobol': 0,
            'invalid_cobol': 0,
            'invalid_details': [],
            'no_cobol_files': 0
        }

        print(f"\nValidating COBOL files in {self.codenet_path}")
        print(f"Found {len(pid_folders)} PID folders")
        print("="*80)

        for i, pid_folder in enumerate(pid_folders, 1):
            problem_id = pid_folder.name

            # Check if COBOL folder exists
            cobol_folder = self.find_cobol_folder(pid_folder)

            if not cobol_folder:
                print(pid_folder, 'no cobol folder')
                stats['no_cobol_folder'] += 1
                continue

            stats['has_cobol_folder'] += 1

            # Find COBOL files
            cobol_files = self.find_cobol_files(cobol_folder)

            if not cobol_files:
                stats['no_cobol_files'] += 1
                print(f"[{i}/{len(pid_folders)}] {problem_id}:  COBOL folder exists but no .cob files")
                continue

            # Validate each COBOL file (usually just one)
            all_valid = True
            for cobol_file in cobol_files:
                result = self.validate_file(cobol_file)

                if not result['valid']:
                    all_valid = False
                    stats['invalid_details'].append({
                        'problem_id': problem_id,
                        'file': cobol_file.name,
                        'detected_language': result['detected_language'],
                        'preview': result.get('preview', '')
                    })

                    print(f"[{i}/{len(pid_folders)}] {problem_id}:  INVALID - "
                          f"Detected as {result['detected_language']}")
                    print(f"    File: {cobol_file.name}")
                    print(f"    Preview: {result.get('preview', 'N/A')}")

            if all_valid:
                stats['valid_cobol'] += 1
                if show_valid:
                    print(f"[{i}/{len(pid_folders)}] {problem_id}:  Valid COBOL")
            else:
                stats['invalid_cobol'] += 1

        print("\n" + "="*80)
        print("SUMMARY")
        print("="*80)
        print(f"Total PID folders:              {stats['total_pids']}")
        print(f"Has COBOL folder:               {stats['has_cobol_folder']}")
        print(f"No COBOL folder:                {stats['no_cobol_folder']}")
        print(f"No .cob files in COBOL folder:  {stats['no_cobol_files']}")
        print()
        print(f"Valid COBOL files:              {stats['valid_cobol']}")
        print(f"Invalid COBOL files:            {stats['invalid_cobol']}")

        if stats['has_cobol_folder'] > 0:
            valid_pct = (stats['valid_cobol'] / stats['has_cobol_folder']) * 100
            print(f"Validity rate:                  {valid_pct:.1f}%")

        print("="*80)

        return stats

    def get_invalid_pids(self):
        """Get list of PIDs with invalid COBOL files"""
        pid_folders = self.get_pid_folders()
        invalid_pids = []

        for pid_folder in pid_folders:
            problem_id = pid_folder.name
            cobol_folder = self.find_cobol_folder(pid_folder)

            if not cobol_folder:
                continue

            cobol_files = self.find_cobol_files(cobol_folder)

            for cobol_file in cobol_files:
                result = self.validate_file(cobol_file)
                if not result['valid']:
                    invalid_pids.append(problem_id)
                    break

        return invalid_pids

    def save_invalid_list(self, output_file='invalid_cobol_pids.txt'):
        """Save list of invalid PIDs to a file"""
        invalid_pids = self.get_invalid_pids()

        with open(output_file, 'w') as f:
            for pid in invalid_pids:
                f.write(f"{pid}\n")

        print(f"\nSaved {len(invalid_pids)} invalid PIDs to {output_file}")
        return invalid_pids


if __name__ == "__main__":
    CodeNet_Dataset_PATH = 'CodeNet_Dataset'

    validator = CobolValidator(CodeNet_Dataset_PATH)

    stats = validator.validate_all(show_valid=False)

    # Get list of invalid PIDs
    invalid_pids = validator.get_invalid_pids()
    print(f"\nFound {len(invalid_pids)} invalid PIDs")

    # Save invalid PIDs to file
    validator.save_invalid_list('invalid_cobol_pids.txt')




>  Compiling with GNUCobol and making sure the tests pass, to get the finalized dataset.


In [ ]:
import subprocess
import os
import json



# Install GnuCOBOL if not already installed
print(" Installing GnuCOBOL...")
os.system("apt-get update -qq")
os.system("apt-get install -y -qq gnucobol")
print("GnuCOBOL installed\n")

# Configuration
CODENET_DIR = 'CodeNet_Dataset'
TEMP_DIR = 'temp_cobol_direct'
RESULTS_FILE = 'cobol_correctness_results.json'

os.makedirs(TEMP_DIR, exist_ok=True)

def test_cobol_directly(problem_id):
    """Compile and run COBOL directly with GnuCOBOL"""

    print(f"Testing: {problem_id}...", end=" ")

    # Get paths
    cobol_dir = os.path.join(CODENET_DIR, problem_id, 'COBOL')
    input_file = os.path.join(CODENET_DIR, problem_id, 'input.txt')
    output_file = os.path.join(CODENET_DIR, problem_id, 'output.txt')

    # Check files exist
    if not os.path.exists(cobol_dir):
        print(" no_cobol_folder")
        return {'status': 'no_cobol_folder', 'error': None}

    if not os.path.exists(input_file) or not os.path.exists(output_file):
        print(" no_test_files")
        return {'status': 'no_test_files', 'error': None}

    # Get first COBOL file
    cobol_files = sorted([f for f in os.listdir(cobol_dir) if f.endswith('.cob')])
    if not cobol_files:
        print(" no_cobol_files")
        return {'status': 'no_cobol_files', 'error': None}

    cobol_file = cobol_files[0]
    cobol_path = os.path.join(cobol_dir, cobol_file)

    # Read test data
    with open(input_file, 'r') as f:
        test_input = f.read()
    with open(output_file, 'r') as f:
        expected_output = f.read()

    # Copy COBOL to temp
    temp_cobol = os.path.join(TEMP_DIR, cobol_file)
    import shutil
    shutil.copy(cobol_path, temp_cobol)

    # Compile with GnuCOBOL
    executable_name = cobol_file.replace('.cob', '')

    try:
        compile_result = subprocess.run(
            ['cobc', '-x', '-free', '-o', executable_name, temp_cobol],
            capture_output=True,
            text=True,
            cwd=TEMP_DIR,
            timeout=5
        )

        if compile_result.returncode != 0:
            print(" compile_failed")
            return {
                'status': 'compile_failed',
                'cobol_file': cobol_file,
                'error': compile_result.stderr,
                'stdout': compile_result.stdout
            }

        # Run the compiled COBOL program
        executable_path = os.path.join(TEMP_DIR, executable_name)

        run_result = subprocess.run(
            [executable_path],
            input=test_input,
            capture_output=True,
            text=True,
            cwd=TEMP_DIR,
            timeout=5
        )

        if run_result.returncode != 0:
            print(" execution_failed")
            return {
                'status': 'execution_failed',
                'cobol_file': cobol_file,
                'error': run_result.stderr,
                'stdout': run_result.stdout,
                'exit_code': run_result.returncode
            }

        actual_output = run_result.stdout.strip()
        expected_output = expected_output.strip()

        # Compare outputs
        actual_lines = [line.strip() for line in actual_output.split('\n')]
        expected_lines = [line.strip() for line in expected_output.split('\n')]

        if actual_lines == expected_lines:
            print(" PASSED")
            return {
                'status': 'passed',
                'cobol_file': cobol_file,
                'error': None
            }
        else:
            print(" output_mismatch")
            return {
                'status': 'output_mismatch',
                'cobol_file': cobol_file,
                'expected_output': expected_output,
                'actual_output': actual_output,
                'error': None
            }

    except subprocess.TimeoutExpired:
        print(" timeout")
        return {
            'status': 'timeout',
            'cobol_file': cobol_file,
            'error': 'Program execution exceeded 5 second timeout'
        }
    except Exception as e:
        print(f" error: {str(e)[:50]}")
        return {
            'status': 'error',
            'cobol_file': cobol_file if 'cobol_file' in locals() else 'unknown',
            'error': str(e)
        }


# Test specific problems
TEST_PIDS = sorted([
        d for d in os.listdir(CODENET_DIR)
        if os.path.isdir(os.path.join(CODENET_DIR, d))
    ])

results = {}

for pid in TEST_PIDS:
    result = test_cobol_directly(pid)
    results[pid] = result

    # Clean up temp files
    for f in os.listdir(TEMP_DIR):
        file_path = os.path.join(TEMP_DIR, f)
        if os.path.isfile(file_path):
            os.remove(file_path)

# Save detailed results to JSON
with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n Detailed results saved to: {RESULTS_FILE}")

# Summary
print("\n" + "="*80)
print("COBOL CORRECTNESS SUMMARY")
print("="*80)

# Count by status
status_counts = {}
for r in results.values():
    status = r['status']
    status_counts[status] = status_counts.get(status, 0) + 1

passed = status_counts.get('passed', 0)
total = len(TEST_PIDS)

print(f"\n Correct COBOL: {passed}/{total} ({passed/total*100:.1f}%)")

print("\nBreakdown by status:")
for status, count in sorted(status_counts.items()):
    symbol = "Y" if status == 'passed' else "N"
    print(f"  {symbol} {status}: {count} ({count/total*100:.1f}%)")

# Cleanup
if os.path.exists(TEMP_DIR):
    import shutil
    shutil.rmtree(TEMP_DIR)

print(f"\n Testing complete!")
print(f" Download results: {RESULTS_FILE}")